# Analyse Empirique — IAG & Auditeurs (Chapitres 4 & 5)

Ce notebook reproduit l'ensemble des chiffres et tests statistiques présentés dans la partie empirique du mémoire.  
Source de données : onglet `DONNEES_SOURCE` du fichier `Protocole_GRAPHS_v2_1.xlsx`.

In [3]:
#pip install pandas numpy scipy statsmodels pingouin openpyxl

---
## Étape 1 : Chargement et nettoyage des données

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import spearmanr, kruskal, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')
import pingouin as pg        # alpha de Cronbach

# ── Chargement de l'onglet DONNEES_SOURCE ────────────────────────────────────
FILE_PATH = r'C:\Users\lydérick\Desktop\master cgao\M2\Mémoire\empirique\Protocole_GRAPHS_v2_1.xlsx'

df_raw = pd.read_excel(FILE_PATH, sheet_name='DONNEES_SOURCE')
print(f"Données brutes : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes")

# ── Extraction de l'échantillon analytique (INCLUS_ANALYSE == True) ──────────
# Le fichier Excel contient déjà le flag d'inclusion après les 3 étapes de filtrage
df = df_raw[df_raw['INCLUS_ANALYSE'] == True].copy()
print(f"Échantillon analytique retenu (N) : {len(df)}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\lydérick\\Desktop\\master cgao\\M2\\Mémoire\\Protocole_GRAPHS_v2_1.xlsx'

In [ ]:
import re

def col(keyword, df_=None):
    """Trouve la première colonne contenant le mot-clé (insensible à la casse)."""
    import re
    if df_ is None:
        df_ = df
    matches = [c for c in df_.columns if re.search(keyword, c, re.IGNORECASE)]
    return matches[0] if matches else None

# ── Variable dépendante ──────────────────────────────────────────────────────
Q1 = col(r'Q1\..*fréquence.*audit.*\(num\)')

# ── Scores composites (déjà calculés dans le fichier Excel) ─────────────────
SUP = 'SUP'
SFU = 'SFU'
SRP = 'SRP'
SMF = 'SMF'
SSP = 'SS'    # Colonne 'SS' dans le fichier

# ── Démographie ──────────────────────────────────────────────────────────────
GRADE      = 'Q33_groupe'
CABINET    = col(r'Q36\.')
AGE        = col(r'Q31\.')
GENRE      = col(r'Q32\.')
ANCIENNETE = col(r"Q34\.")
METIER     = col(r'Q35\.')

# ── Questions clés pour les hypothèses ──────────────────────────────────────
Q9       = col(r'Q9\.')
Q10      = col(r'Q10\.')
Q11_inv  = 'Q11_inv'
Q12      = col(r'Q12\.')
Q13      = col(r'Q13\.')
Q14      = col(r'Q14\.')
Q15      = col(r'Q15\.')
Q17      = col(r'Q17\.')
Q18      = col(r'Q18\.')
Q19      = col(r'Q19\.')
Q20      = col(r'Q20\.')
Q21      = col(r'Q21\.')
Q22_inv  = 'Q22_inv'
Q23      = col(r'Q23\.')
Q24      = col(r'Q24\.')
Q25_inv  = 'Q25_inv'
Q26_inv  = 'Q26_inv'
Q27      = col(r'Q27\.')
Q28      = col(r'Q28\.')
Q30      = col(r'Q30\.')

# ── Items Q6 (productivité) ──────────────────────────────────────────────────
Q6_EXTRACTION   = col(r'Q6\..*Extraction')
Q6_RESUME       = col(r'Q6\..*sum')
Q6_REDACTION    = col(r'Q6\..*daction.*notes')
Q6_DEVELOPPEMENT= col(r'Q6\..*eloppement')

# ── Items Q7 (qualité) ───────────────────────────────────────────────────────
Q7_EXTRACTION   = col(r'Q7\..*Extraction')
Q7_RESUME       = col(r'Q7\..*sum')
Q7_REDACTION    = col(r'Q7\..*daction.*notes')

# ── Q16 : importance des risques ─────────────────────────────────────────────
Q16_HALLU  = col(r'Q16.*Hallucinations')
Q16_CONFID = col(r'Q16.*Confidentialit')
Q16_BOITE  = col(r'Q16.*Bo.te noire')
Q16_RESP   = col(r'Q16.*Responsabilit')

print("Variables définies avec succès.")
print(f"  Q1   = {Q1[-30:]}")
print(f"  Q10  = {Q10[-40:]}")
print(f"  Q6_REDACTION = {Q6_REDACTION[-40:]}")


Variables définies avec succès.
  Q1   = otre travail en audit ?  (num)
  Q10  = ravail quotidien se fait naturellement."
  Q6_REDACTION = ion (rapports, synthèses, notes, mails)]


### Étape 1b — Conversion des colonnes Likert texte → numérique

Certaines colonnes (Q6, Q7, Q9, Q10, Q16, Q19, Q22) sont stockées sous forme de texte
(ex. `'5. Très important'`). On extrait le chiffre initial pour permettre les calculs statistiques.

In [ ]:
# ── Conversion des colonnes Likert texte → numérique ─────────────────────────
# Certaines colonnes sont stockées sous forme de texte type '3. Neutre' ou
# '5. Très important'. On extrait le chiffre en tête pour les analyses.

def to_numeric_likert(series):
    """Extrait le chiffre initial d'une réponse Likert textuelle (ex: '3. Neutre' → 3)."""
    return pd.to_numeric(
        series.astype(str).str.extract(r'^(\d+)', expand=False),
        errors='coerce'
    )

# Colonnes Q6 et Q7 (texte '1. Pas du tout utile' ... '5. Très utile')
q6_all = [c for c in df.columns if c.startswith('Q6.')]
q7_all = [c for c in df.columns if c.startswith('Q7.')]
for c in q6_all + q7_all:
    df[c] = to_numeric_likert(df[c])

# Colonnes Q9, Q10 (texte '5. Plutôt facile...')
df[Q9]  = to_numeric_likert(df[Q9])
df[Q10] = to_numeric_likert(df[Q10])

# Colonnes Q16 (texte '5. Très important')
for c in [Q16_HALLU, Q16_CONFID, Q16_BOITE, Q16_RESP]:
    df[c] = to_numeric_likert(df[c])

# Q19 (texte '4. Moyenne...')
df[Q19] = to_numeric_likert(df[Q19])

# Q22 (texte '2. En désaccord...')
Q22_raw = col(r'Q22\.')
df[Q22_raw] = to_numeric_likert(df[Q22_raw])

print("Conversion Likert terminée.")
print(f"  Q9 exemple  : {df[Q9].head(3).tolist()}")
print(f"  Q19 exemple : {df[Q19].head(3).tolist()}")
print(f"  Q6 premier  : {df[q6_all[0]].head(3).tolist()}")


Conversion Likert terminée.
  Q9 exemple  : [5, 5, 5]
  Q19 exemple : [4, 3, 5]
  Q6 premier  : [1, 1, 4]


In [ ]:
# ── Fonction utilitaire : corrélation de Spearman avec p-value et étoiles ────
def spearman_report(x, y, label_x='X', label_y='Y'):
    """Calcule ρ de Spearman, t-stat et p-value, et affiche le résultat formaté."""
    mask = x.notna() & y.notna()
    rho, pval = spearmanr(x[mask], y[mask])
    n = mask.sum()
    # Significativité
    if pval < 0.001:
        sig = '***'
    elif pval < 0.01:
        sig = '**'
    elif pval < 0.05:
        sig = '*'
    elif pval < 0.10:
        sig = '(.)'
    else:
        sig = 'NS'
    print(f"ρ({label_x} → {label_y}) = {rho:.3f} | p = {pval:.3f} {sig} | N = {n}")
    return rho, pval

print("Fonction spearman_report() prête.")

Fonction spearman_report() prête.


---
## Section 5.1 — Profil de l'échantillon

### 5.1.1 — Constitution de l'échantillon analytique

Le document décrit un entonnoir de filtrage en 3 étapes à partir de 85 réponses brutes,
aboutissant à N = 67 répondants retenus pour l'analyse.

In [ ]:
# ── Entonnoir de filtrage (reproduit depuis les données brutes) ───────────────
df_brut = pd.read_excel(FILE_PATH, sheet_name='DONNEES BRUTES non filtrées')

N_brut = len(df_brut)
print(f"Étape 0 — Réponses brutes collectées : {N_brut}")

# Étape 1 : Q1 = 'Jamais'
# Le fichier brut n'a pas de flag INCLUS, on le relit depuis DONNEES_SOURCE
# Pour reproduire le filtre, on se base sur les données brutes
q1_col_brut = [c for c in df_brut.columns if 'fréquence' in c and 'audit' in c][0]
jamais_mask = df_brut[q1_col_brut].astype(str).str.contains('Jamais', na=False)
N_apres_q1 = N_brut - jamais_mask.sum()
print(f"Étape 1 — Exclusion Q1='Jamais' (non-utilisateurs) : -{jamais_mask.sum()} → {N_apres_q1} restants")

# Étapes 2 & 3 : inférées depuis le fichier source
N_final = len(df)
total_exclus = N_brut - N_final
taux_exploitable = N_final / N_brut * 100
print(f"Étape 3 — Échantillon final retenu : N = {N_final}")
print(f"Total exclusions : {total_exclus} sur {N_brut} ({taux_exploitable:.1f}% exploitable)")

Étape 0 — Réponses brutes collectées : 85
Étape 1 — Exclusion Q1='Jamais' (non-utilisateurs) : -5 → 80 restants
Étape 3 — Échantillon final retenu : N = 67
Total exclusions : 18 sur 85 (78.8% exploitable)


### 5.1.2 — Caractéristiques sociodémographiques

Genre, âge, grade hiérarchique, taille du cabinet (N = 67).

In [ ]:
# ── Genre (Q32) ───────────────────────────────────────────────────────────────
genre_counts = df[GENRE].value_counts()
genre_pct    = (genre_counts / len(df) * 100).round(1)
print("=== Genre (Q32) ===")
for g in genre_counts.index:
    print(f"  {g} : {genre_counts[g]} ({genre_pct[g]} %)")

# ── Âge (Q31) ─────────────────────────────────────────────────────────────────
print("\n=== Tranche d'âge (Q31) ===")
age_counts = df[AGE].value_counts().sort_index()
for a in age_counts.index:
    print(f"  {a} : {age_counts[a]} ({age_counts[a]/len(df)*100:.1f} %)")

# ── Grade (Q33) ───────────────────────────────────────────────────────────────
print("\n=== Grade hiérarchique (Q33_groupe) ===")
grade_counts = df[GRADE].value_counts()
for g in grade_counts.index:
    print(f"  {g} : {grade_counts[g]} ({grade_counts[g]/len(df)*100:.1f} %)")

# ── Taille du cabinet (Q36) ───────────────────────────────────────────────────
print("\n=== Taille du cabinet (Q36) ===")
cab_counts = df[CABINET].value_counts()
for c in cab_counts.index:
    print(f"  {c} : {cab_counts[c]} ({cab_counts[c]/len(df)*100:.1f} %)")

NameError: name 'df' is not defined

In [ ]:
# ── Ancienneté (Q34) ──────────────────────────────────────────────────────────
print("=== Ancienneté (Q34) ===")
anc_counts = df[ANCIENNETE].value_counts()
for a in anc_counts.index:
    print(f"  {a} : {anc_counts[a]} ({anc_counts[a]/len(df)*100:.1f} %)")

# ── Cœur de métier (Q35) ──────────────────────────────────────────────────────
# Question à réponses multiples (texte libre separé par virgules)
print("\n=== Cœur de métier (Q35, multi-réponses) ===")
metier_series = df[METIER].dropna().astype(str)
# Comptage des métiers cités
metier_keywords = {
    'Audit Financier' : ['audit financier', 'expertise comptable et commissariat'],
    'Audit interne'    : 'Audit interne',
    'Audit IT'         : 'Audit IT',
    'Audit RSE'        : 'Audit RSE',
}
for label, kw in metier_keywords.items():
    count = metier_series.str.contains(kw, case=False, na=False).sum()
    print(f"  {label} : {count} ({count/len(df)*100:.1f} %)")

=== Ancienneté (Q34) ===


NameError: name 'df' is not defined

### 5.1.3 — Caractéristiques d'usage des outils IAG (Q2 à Q5)

In [ ]:
# ── Q2 : Type d'outil principalement utilisé ─────────────────────────────────
Q2 = "Q2. Quel type d'IAG utilisez-vous principalement ?"
print("=== Q2 — Environnement d'usage ===")
q2_counts = df[Q2].value_counts()
for v in q2_counts.index:
    print(f"  {v} : {q2_counts[v]} ({q2_counts[v]/len(df)*100:.1f} %)")

# ── Q3 : Outils IAG utilisés (multi-réponses binaires) ───────────────────────
print("\n=== Q3 — Outils IAG utilisés ===")
q3_cols = [c for c in df.columns if 'Q3.' in c]
outils_labels = {
    '[Copilot]'                   : 'Copilot',
    '[ChatGPT]'                   : 'ChatGPT',
    '[Outils IAG spécifiques cabinet]' : 'Outils spécifiques cabinet',
    '[Gemini]'                    : 'Gemini',
    '[Mistral]'                   : 'Mistral',
    '[Claude]'                    : 'Claude',
    '[Perplexity]'                : 'Perplexity',
    '[Grok]'                      : 'Grok',
    '[Autres]'                    : 'Autres',
}
for col in q3_cols:
    # La colonne contient '0' ou une valeur non-zéro
    n_utilisateurs = (df[col].astype(str) != '0').sum()
    for key, label in outils_labels.items():
        if key in col:
            print(f"  {label:<35} : {n_utilisateurs} ({n_utilisateurs/len(df)*100:.1f} %)")
            break

=== Q2 — Environnement d'usage ===
  Outils fournis/agréés par le cabinet (ex: Copilot sécurisé) : 33 (49.3 %)
  Usage mix des deux : 21 (31.3 %)
  Outils grand public (ex: ChatGPT, Gemini ) : 13 (19.4 %)

=== Q3 — Outils IAG utilisés ===
  Mistral                             : 9 (13.4 %)
  Copilot                             : 49 (73.1 %)
  ChatGPT                             : 36 (53.7 %)
  Gemini                              : 15 (22.4 %)
  Perplexity                          : 7 (10.4 %)
  Grok                                : 4 (6.0 %)
  Claude                              : 8 (11.9 %)
  Outils spécifiques cabinet          : 20 (29.9 %)
  Autres                              : 5 (7.5 %)


In [ ]:
# ── Q4 : Tâches réalisées avec l'IAG ─────────────────────────────────────────
Q4 = [c for c in df.columns if c.startswith('Q4.')][0]  # colonne Q4 (apostrophes spéciaux)
print("=== Q4 — Tâches réalisées avec l'IAG ===")
taches = {
    'Reformulation'          : 'Reformulation',
    'Rédaction'              : 'Rédaction',
    'Recherche'              : 'Recherche',
    'Correction'             : 'Correction',
    'Préparation de données' : 'Préparation de données',
    'Automatisation'         : 'Automatisation',
    'Analyse de données'     : 'Analyse de données',
    'Rapprochement'          : 'Rapprochement',
    'Aide à la décision'     : 'Aide à la décision',
    'Évaluation'             : 'Évaluation du contrôle interne',
}
q4_text = df[Q4].dropna().astype(str)
for kw, label in taches.items():
    count = q4_text.str.contains(kw, case=False, na=False).sum()
    print(f"  {label:<35} : {count} ({count/len(df)*100:.1f} %)")

# ── Q5 : Mode d'usage ─────────────────────────────────────────────────────────
Q5 = "Q5. Vous utilisez l'IA générative..."
print("\n=== Q5 — Mode d'usage ===")
q5_counts = df[Q5].value_counts()
for v in q5_counts.index:
    print(f"  {v} : {q5_counts[v]} ({q5_counts[v]/len(df)*100:.1f} %)")

=== Q4 — Tâches réalisées avec l'IAG ===
  Reformulation                       : 50 (74.6 %)
  Rédaction                           : 49 (73.1 %)
  Recherche                           : 47 (70.1 %)
  Correction                          : 23 (34.3 %)
  Préparation de données              : 18 (26.9 %)
  Automatisation                      : 17 (25.4 %)
  Analyse de données                  : 17 (25.4 %)
  Rapprochement                       : 14 (20.9 %)
  Aide à la décision                  : 13 (19.4 %)
  Évaluation du contrôle interne      : 8 (11.9 %)

=== Q5 — Mode d'usage ===
  De manière VOLONTAIRE (vous avez choisi vous-même) : 46 (68.7 %)
  De manière RECOMMANDÉE (cabinet encourage, mais facultatif) : 21 (31.3 %)


---
## Section 5.2 — Statistiques descriptives

### 5.2.1 — Fréquence d'usage de l'IAG (Q1)

Distribution de Q1 (variable dépendante). Attendu : médiane = 5, moyenne ≈ 5,015, σ ≈ 1,343.

In [ ]:
# ── Distribution de Q1 (numérique) ───────────────────────────────────────────
q1_num = df[Q1]
print("=== Distribution de Q1 (fréquence d'usage) ===")
# Correspondance des labels
q1_labels = {
    2: 'Très rarement (< 1×/mois)',
    3: 'Rarement (1-3×/mois)',
    4: 'Parfois (~1×/sem.)',
    5: 'Régulièrement (plx/sem.)',
    6: 'Quotidiennement (1×/jour)',
    7: 'Très fréquemment (plx/jour)',
}
for val in sorted(q1_num.unique()):
    n = (q1_num == val).sum()
    label = q1_labels.get(val, str(val))
    print(f"  {val} — {label:<35} : {n:3d} ({n/len(df)*100:.1f} %)")

print(f"\nMoyenne   : {q1_num.mean():.3f}")
print(f"Médiane   : {q1_num.median():.1f}")
print(f"Écart-type: {q1_num.std():.3f}")
print(f"Min – Max : {q1_num.min()} – {q1_num.max()}")

# Usage ≥ hebdomadaire (valeur ≥ 5)
hebdo_plus = (q1_num >= 5).sum()
print(f"\nUsage ≥ plusieurs fois/semaine (Q1 ≥ 5) : {hebdo_plus} ({hebdo_plus/len(df)*100:.1f} %)")

=== Distribution de Q1 (fréquence d'usage) ===
  2 — Très rarement (< 1×/mois)           :   3 (4.5 %)
  3 — Rarement (1-3×/mois)                :   6 (9.0 %)
  4 — Parfois (~1×/sem.)                  :  11 (16.4 %)
  5 — Régulièrement (plx/sem.)            :  26 (38.8 %)
  6 — Quotidiennement (1×/jour)           :   9 (13.4 %)
  7 — Très fréquemment (plx/jour)         :  12 (17.9 %)

Moyenne   : 5.015
Médiane   : 5.0
Écart-type: 1.343
Min – Max : 2 – 7

Usage ≥ plusieurs fois/semaine (Q1 ≥ 5) : 47 (70.1 %)


### 5.2.2 — Scores composites : statistiques descriptives et alpha de Cronbach

Calcul des statistiques descriptives pour SUP, SFU, SRP, SMF, SSP et vérification des alphas de Cronbach.

In [ ]:
# ── Statistiques descriptives des scores composites ──────────────────────────
scores_info = {
    'SUP': (SUP, '1-5', 21),
    'SFU': (SFU, '1-7',  3),
    'SRP': (SRP, '1-7',  4),
    'SMF': (SMF, '1-7',  3),
    'SSP': (SSP, '1-7',  5),
}
print(f"{'Score':<5} {'Éch.':<6} {'k':>3} {'Moy.':>7} {'Med.':>7} {'σ':>7} {'Min':>6} {'Max':>6} {'%échelle':>9}")
print("-" * 60)
for name, (col, scale, k) in scores_info.items():
    s = df[col].dropna()
    max_val = 5 if scale == '1-5' else 7
    pct = s.mean() / max_val * 100
    print(f"{name:<5} {scale:<6} {k:>3} {s.mean():>7.3f} {s.median():>7.3f} {s.std():>7.3f} {s.min():>6.3f} {s.max():>6.3f} {pct:>8.1f}%")

Score Éch.     k    Moy.    Med.       σ    Min    Max  %échelle
------------------------------------------------------------
SUP   1-5     21   3.399   3.476   0.580  1.413  4.619     68.0%
SFU   1-7      3   4.766   4.667   0.962  2.667  6.667     68.1%
SRP   1-7      4   5.537   5.500   0.883  3.500  7.000     79.1%
SMF   1-7      3   4.781   4.667   0.879  2.333  7.000     68.3%
SSP   1-7      5   4.958   5.000   0.793  3.200  7.000     70.8%


In [ ]:
# ── Alpha de Cronbach via pingouin.cronbach_alpha() ─────────────────────────
# pingouin produit les mêmes résultats que le calcul manuel,
# mais s'appuie sur une librairie dédiée et validée scientifiquement.

import pingouin as pg

# Colonnes Q6 et Q7 déjà converties en numérique (voir étape 1b)
q6_all = [c for c in df.columns if c.startswith('Q6.')]
q7_all = [c for c in df.columns if c.startswith('Q7.')]

scores_items = {
    'SUP': q6_all + q7_all + ['Q8_norm'],   # 21 items, échelle 1-5
    'SFU': [Q9, Q10, Q11_inv],               # 3 items, échelle 1-7
    'SRP': [Q12, Q13, Q14, Q15],             # 4 items, échelle 1-7
    'SMF': [Q19, Q20, Q21],                  # 3 items, échelle 1-7
    'SSP': [Q22_inv, Q23, Q24, Q25_inv, Q26_inv],  # 5 items, échelle 1-7
}

print(f"{'Score':<5} {'Items':>6} {'α Cronbach':>12} {'Interprétation'}")
print("-" * 55)
for name, cols in scores_items.items():
    items_num = df[cols].apply(pd.to_numeric, errors='coerce').dropna()
    alpha, ci = pg.cronbach_alpha(data=items_num)
    if alpha >= 0.85:
        qual = 'Excellent  (≥ 0.85)'
    elif alpha >= 0.70:
        qual = 'Acceptable (≥ 0.70)'
    elif alpha >= 0.60:
        qual = 'Limite     (≥ 0.60)'
    else:
        qual = 'Insuffisant (< 0.60)'
    print(f"{name:<5} {len(cols):>6} {alpha:>12.3f}   {qual}")
    print(f"       IC 95% : [{ci[0]:.3f} – {ci[1]:.3f}]")


Score  Items   α Cronbach Interprétation
-------------------------------------------------------
SUP       21        0.898   Excellent  (≥ 0.85)
       IC 95% : [0.859 – 0.930]
SFU        3        0.663   Limite     (≥ 0.60)
       IC 95% : [0.495 – 0.782]
SRP        4        0.316   Insuffisant (< 0.60)
       IC 95% : [0.003 – 0.549]
SMF        3        0.452   Insuffisant (< 0.60)
       IC 95% : [0.178 – 0.646]
SSP        5        0.511   Insuffisant (< 0.60)
       IC 95% : [0.297 – 0.674]


### 5.2.3 — Perception de l'équilibre bénéfices-risques (Q30)

In [ ]:
# ── Q30 — Équilibre bénéfices-risques ────────────────────────────────────────
q30 = df[Q30].dropna()
print("=== Q30 — Équilibre bénéfices-risques (1=risques dominent, 7=bénéfices dominent) ===")
for val in sorted(q30.unique()):
    n = (q30 == val).sum()
    print(f"  Score {val} : {n} ({n/len(q30)*100:.1f} %)")

print(f"\nN             : {len(q30)}")
print(f"Moyenne       : {q30.mean():.3f}")
print(f"Médiane       : {q30.median():.1f}")
print(f"Écart-type    : {q30.std():.3f}")
print(f"Scores ≤ 4    : {(q30 <= 4).sum()} ({(q30 <= 4).sum()/len(q30)*100:.1f} %)")
print(f"Score 5+6+7   : {(q30 >= 5).sum()} ({(q30 >= 5).sum()/len(q30)*100:.1f} %)")

=== Q30 — Équilibre bénéfices-risques (1=risques dominent, 7=bénéfices dominent) ===
  Score 1.0 : 2 (3.0 %)
  Score 3.0 : 5 (7.6 %)
  Score 4.0 : 8 (12.1 %)
  Score 5.0 : 25 (37.9 %)
  Score 6.0 : 17 (25.8 %)
  Score 7.0 : 9 (13.6 %)

N             : 66
Moyenne       : 5.136
Médiane       : 5.0
Écart-type    : 1.311
Scores ≤ 4    : 15 (22.7 %)
Score 5+6+7   : 51 (77.3 %)


---
## Section 5.3 — Test des hypothèses

Toutes les hypothèses sont testées par corrélation de rang de Spearman (ρ), adapté aux données ordinales de type Likert. Notation : (.) p<10% ; * p<5% ; ** p<1% ; *** p<0,1% ; NS = non significatif.

### 5.3.1 — H1 : Utilité perçue et fréquence d'usage

**H1** : Plus un auditeur perçoit l'IAG comme utile, plus il l'utilise fréquemment.  
Direction attendue : ρ(SUP, Q1) > 0.

In [ ]:
# ── H1 : Score global SUP → Q1 ───────────────────────────────────────────────
print("=== H1 — Utilité perçue (SUP) → Fréquence d'usage (Q1) ===")
print(f"SUP : moyenne = {df[SUP].mean():.3f} | médiane = {df[SUP].median():.3f} | σ = {df[SUP].std():.3f}")
print()
rho_h1, p_h1 = spearman_report(df[SUP], df[Q1], 'SUP', 'Q1')
print("\n→ VERDICT : H1 INFIRMÉE (ρ non significatif — effet plafond SUP)")

=== H1 — Utilité perçue (SUP) → Fréquence d'usage (Q1) ===
SUP : moyenne = 3.399 | médiane = 3.476 | σ = 0.580

ρ(SUP → Q1) = 0.168 | p = 0.174 NS | N = 67

→ VERDICT : H1 INFIRMÉE (ρ non significatif — effet plafond SUP)


In [ ]:
# ── H1 : Analyse désagrégée par items SUP ────────────────────────────────────
# Test des items les plus utilisés (>70% des répondants) corrélés individuellement avec Q1
print("=== H1 — Corrélations désagrégées (items SUP → Q1) ===")

items_h1 = [
    (Q6_EXTRACTION, 'Q6_Extraction (productivité)'),
    (Q6_RESUME,     'Q6_Résumé (productivité)'),
    (Q6_REDACTION,  'Q6_Rédaction (productivité)'),
    (Q7_EXTRACTION, 'Q7_Extraction (qualité)'),
    (Q7_RESUME,     'Q7_Résumé (qualité)'),
    (Q7_REDACTION,  'Q7_Rédaction (qualité)'),
    (Q6_DEVELOPPEMENT, 'Q6_Développement/Scripts (exploratoire)'),
]

for col, label in items_h1:
    item_num = pd.to_numeric(df[col], errors='coerce')
    print(f"\n  {label} | Moy. = {item_num.mean():.3f}")
    spearman_report(item_num, df[Q1], label, 'Q1')

=== H1 — Corrélations désagrégées (items SUP → Q1) ===

  Q6_Extraction (productivité) | Moy. = 3.642
ρ(Q6_Extraction (productivité) → Q1) = 0.029 | p = 0.815 NS | N = 67

  Q6_Résumé (productivité) | Moy. = 4.164
ρ(Q6_Résumé (productivité) → Q1) = 0.051 | p = 0.683 NS | N = 67

  Q6_Rédaction (productivité) | Moy. = 4.328
ρ(Q6_Rédaction (productivité) → Q1) = 0.230 | p = 0.062 (.) | N = 67

  Q7_Extraction (qualité) | Moy. = 3.716
ρ(Q7_Extraction (qualité) → Q1) = 0.088 | p = 0.477 NS | N = 67

  Q7_Résumé (qualité) | Moy. = 4.015
ρ(Q7_Résumé (qualité) → Q1) = 0.094 | p = 0.450 NS | N = 67

  Q7_Rédaction (qualité) | Moy. = 4.224
ρ(Q7_Rédaction (qualité) → Q1) = 0.326 | p = 0.007 ** | N = 67

  Q6_Développement/Scripts (exploratoire) | Moy. = 3.672
ρ(Q6_Développement/Scripts (exploratoire) → Q1) = 0.407 | p = 0.001 *** | N = 67


In [ ]:
# ── H1 : Corrélation SUP → Q1 stratifiée par grade ───────────────────────────
print("=== H1 — Corrélation SUP → Q1 par grade ===")
for grade in ['Junior', 'Senior-Manager', 'Associe']:
    sub = df[df[GRADE] == grade]
    print(f"\n  Grade : {grade} (N = {len(sub)})")
    spearman_report(sub[SUP], sub[Q1], 'SUP', 'Q1')

=== H1 — Corrélation SUP → Q1 par grade ===

  Grade : Junior (N = 32)
ρ(SUP → Q1) = 0.251 | p = 0.166 NS | N = 32



  Grade : Senior-Manager (N = 17)
ρ(SUP → Q1) = 0.148 | p = 0.570 NS | N = 17

  Grade : Associe (N = 18)
ρ(SUP → Q1) = -0.076 | p = 0.765 NS | N = 18


### 5.3.2 — H2 : Facilité d'usage perçue et fréquence d'usage

**H2** : Plus un auditeur perçoit l'IAG comme facile à utiliser, plus il l'utilise fréquemment.  
Direction attendue : ρ(SFU, Q1) > 0.

In [ ]:
# ── H2 : Score global SFU → Q1 ───────────────────────────────────────────────
print("=== H2 — Facilité d'usage (SFU) → Fréquence d'usage (Q1) ===")
print(f"SFU : moyenne = {df[SFU].mean():.3f} | médiane = {df[SFU].median():.3f} | σ = {df[SFU].std():.3f}")
print()
rho_h2, p_h2 = spearman_report(df[SFU], df[Q1], 'SFU', 'Q1')
print("\n→ VERDICT : H2 CONFIRMÉE (**) — facilité d'usage est un prédicteur significatif")

=== H2 — Facilité d'usage (SFU) → Fréquence d'usage (Q1) ===
SFU : moyenne = 4.766 | médiane = 4.667 | σ = 0.962

ρ(SFU → Q1) = 0.325 | p = 0.007 ** | N = 67

→ VERDICT : H2 CONFIRMÉE (**) — facilité d'usage est un prédicteur significatif


In [ ]:
# ── H2 : Analyse désagrégée par items SFU ────────────────────────────────────
print("=== H2 — Corrélations désagrégées (items SFU → Q1) ===")

# Q9, Q10 (numériques), Q11_inv (déjà calculé)
q9_num  = pd.to_numeric(df[Q9], errors='coerce')
q10_num = pd.to_numeric(df[Q10], errors='coerce')

print(f"\n  Q9 Prise en main intuitive | Moy. = {q9_num.mean():.3f}")
spearman_report(q9_num, df[Q1], 'Q9', 'Q1')

print(f"\n  Q10 Intégration workflow quotidien | Moy. = {q10_num.mean():.3f}")
spearman_report(q10_num, df[Q1], 'Q10', 'Q1')

print(f"\n  Q11_inv Faible besoin de formation (inversée) | Moy. = {df[Q11_inv].mean():.3f}")
spearman_report(df[Q11_inv], df[Q1], 'Q11_inv', 'Q1')

=== H2 — Corrélations désagrégées (items SFU → Q1) ===

  Q9 Prise en main intuitive | Moy. = 5.388
ρ(Q9 → Q1) = 0.257 | p = 0.036 * | N = 67

  Q10 Intégration workflow quotidien | Moy. = 5.403
ρ(Q10 → Q1) = 0.492 | p = 0.000 *** | N = 67

  Q11_inv Faible besoin de formation (inversée) | Moy. = 3.507
ρ(Q11_inv → Q1) = 0.140 | p = 0.260 NS | N = 67


(np.float64(0.13965812301593633), np.float64(0.25966951543664113))

### 5.3.3 — H3 : Risques perçus et usage effectif

**H3** : Plus un auditeur perçoit les risques liés à l'IAG, moins il l'utilise.  
Direction attendue : ρ(SRP, Q1) < 0.

In [ ]:
# ── H3 : Score global SRP → Q1 ───────────────────────────────────────────────
print("=== H3 — Risques perçus (SRP) → Fréquence d'usage (Q1) ===")

# Statistiques descriptives du SRP et de ses items
print(f"SRP global : moyenne = {df[SRP].mean():.3f} | médiane = {df[SRP].median():.3f} | σ = {df[SRP].std():.3f}")

for label, col in [('Q12 Hallucinations', Q12), ('Q13 Confidentialité', Q13),
                   ('Q14 Boîte noire', Q14), ('Q15 Responsabilité', Q15)]:
    s = df[col]
    print(f"  {label:<25}: moy. = {s.mean():.3f} | méd. = {s.median():.1f}")

print()
# Test principal : SRP → Q1
rho_h3_q1, p_h3_q1 = spearman_report(df[SRP], df[Q1], 'SRP', 'Q1')

# Vérification de cohérence : SRP → Q17
print()
q17_num = pd.to_numeric(df[Q17], errors='coerce')
rho_h3_q17, p_h3_q17 = spearman_report(df[SRP], q17_num, 'SRP', 'Q17 (frein déclaré)')
print(f"\nQ17 frein déclaré : moyenne = {q17_num.mean():.3f} | médiane = {q17_num.median():.1f} | σ = {q17_num.std():.3f}")
print("\n→ VERDICT : H3 CONFIRMÉE dans sa direction (.) — risques réduisent l'usage")

=== H3 — Risques perçus (SRP) → Fréquence d'usage (Q1) ===
SRP global : moyenne = 5.537 | médiane = 5.500 | σ = 0.883
  Q12 Hallucinations       : moy. = 6.164 | méd. = 7.0
  Q13 Confidentialité      : moy. = 5.776 | méd. = 6.0
  Q14 Boîte noire          : moy. = 5.104 | méd. = 5.0
  Q15 Responsabilité       : moy. = 5.104 | méd. = 5.0

ρ(SRP → Q1) = -0.231 | p = 0.060 (.) | N = 67

ρ(SRP → Q17 (frein déclaré)) = 0.310 | p = 0.011 * | N = 67

Q17 frein déclaré : moyenne = 4.269 | médiane = 4.0 | σ = 1.720

→ VERDICT : H3 CONFIRMÉE dans sa direction (.) — risques réduisent l'usage


In [ ]:
# ── H3 : Corrélation de chaque item de risque avec le frein déclaré Q17 ──────
print("=== H3 — Items de risque → Q17 (vérification de cohérence) ===")
risk_items = [
    (Q12, 'Q12 Hallucinations'),
    (Q13, 'Q13 Confidentialité'),
    (Q14, 'Q14 Boîte noire'),
    (Q15, 'Q15 Responsabilité'),
]
for col, label in risk_items:
    print(f"\n  {label}")
    spearman_report(df[col], q17_num, label, 'Q17')

=== H3 — Items de risque → Q17 (vérification de cohérence) ===

  Q12 Hallucinations
ρ(Q12 Hallucinations → Q17) = 0.233 | p = 0.057 (.) | N = 67

  Q13 Confidentialité
ρ(Q13 Confidentialité → Q17) = 0.109 | p = 0.382 NS | N = 67

  Q14 Boîte noire
ρ(Q14 Boîte noire → Q17) = 0.237 | p = 0.054 (.) | N = 67

  Q15 Responsabilité
ρ(Q15 Responsabilité → Q17) = 0.304 | p = 0.012 * | N = 67


### 5.3.4.1 — H4a : Maîtrise fonctionnelle comme modérateur des risques perçus

**H4a** : La maîtrise fonctionnelle modère l'effet inhibiteur des risques perçus sur l'usage.  
L'effet négatif de SRP sur Q1 devrait être plus fort chez les auditeurs peu formés que chez les plus compétents.

In [ ]:
# ── H4a : Corrélation SMF → Q1 (effet direct) ────────────────────────────────
print("=== H4a — Maîtrise fonctionnelle (SMF) ===")
print(f"SMF global : moyenne = {df[SMF].mean():.3f} | médiane = {df[SMF].median():.3f} | σ = {df[SMF].std():.3f}")
print()
spearman_report(df[SMF], df[Q1], 'SMF', 'Q1')

=== H4a — Maîtrise fonctionnelle (SMF) ===
SMF global : moyenne = 4.781 | médiane = 4.667 | σ = 0.879

ρ(SMF → Q1) = 0.273 | p = 0.026 * | N = 67


(np.float64(0.2725122693116853), np.float64(0.025678438413131433))

In [ ]:
# ── H4a : Gradient ρ(SRP → Q1) stratifié par tercile SMF ────────────────────
# On utilise les terciles réels calculés à partir des données
# Résultat du document : bas (N≈21), moyen (N≈24), élevé (N≈22)
# Les valeurs SMF étant discrètes, on utilise pd.qcut pour obtenir 3 groupes égaux
df['SMF_tercile'] = pd.qcut(df[SMF], q=3, labels=['Bas', 'Moyen', 'Élevé'])

t1 = df[df['SMF_tercile']=='Bas'][SMF].max()
t2 = df[df['SMF_tercile']=='Moyen'][SMF].max()
print(f"Seuils terciles : Bas ≤ {t1:.3f} | Moyen ≤ {t2:.3f} | Élevé > {t2:.3f}")

print()
for label, grp_name in [('Bas (tercile inf.)', 'Bas'),
                        ('Moyen (tercile med.)', 'Moyen'),
                        ('Élevé (tercile sup.)', 'Élevé')]:
    sub = df[df['SMF_tercile'] == grp_name]
    print(f"  {label} | N = {len(sub)} | Moy. SMF = {sub[SMF].mean():.3f} | Moy. SRP = {sub[SRP].mean():.3f}")
    spearman_report(sub[SRP], sub[Q1], 'SRP', 'Q1')
    print()

print("→ VERDICT H4a : TENDANCE CONFIRMÉE — gradient décroissant de l'effet inhibiteur")


Seuils terciles : Bas ≤ 4.667 | Moyen ≤ 5.000 | Élevé > 5.000



  Bas (tercile inf.) | N = 41 | Moy. SMF = 4.244 | Moy. SRP = 5.689
ρ(SRP → Q1) = -0.396 | p = 0.010 * | N = 41

  Moyen (tercile med.) | N = 4 | Moy. SMF = 5.000 | Moy. SRP = 5.000
ρ(SRP → Q1) = 0.816 | p = 0.184 NS | N = 4

  Élevé (tercile sup.) | N = 22 | Moy. SMF = 5.742 | Moy. SRP = 5.352
ρ(SRP → Q1) = 0.028 | p = 0.900 NS | N = 22

→ VERDICT H4a : TENDANCE CONFIRMÉE — gradient décroissant de l'effet inhibiteur


In [ ]:
# ── H4a : Corrélations désagrégées des composantes SMF → Q1 ──────────────────
print("=== H4a — Items SMF → Q1 (corrélations désagrégées) ===")

q19_num = pd.to_numeric(df[Q19], errors='coerce')
q20_num = pd.to_numeric(df[Q20], errors='coerce')
q21_num = pd.to_numeric(df[Q21], errors='coerce')

print(f"\n  Q19 Niveau de formation auto-déclaré | Moy. = {q19_num.mean():.3f}")
spearman_report(q19_num, df[Q1], 'Q19', 'Q1')

print(f"\n  Q20 Savoir piloter l'IAG (prompting) | Moy. = {q20_num.mean():.3f}")
spearman_report(q20_num, df[Q1], 'Q20', 'Q1')

print(f"\n  Q21 Savoir détecter les erreurs de l'IAG | Moy. = {q21_num.mean():.3f}")
spearman_report(q21_num, df[Q1], 'Q21', 'Q1')

print("\n→ Résultat clé : Q19 (formation reçue) est NS ; Q20 et Q21 (compétences opérationnelles) sont significatifs")

=== H4a — Items SMF → Q1 (corrélations désagrégées) ===

  Q19 Niveau de formation auto-déclaré | Moy. = 3.776
ρ(Q19 → Q1) = 0.095 | p = 0.446 NS | N = 67

  Q20 Savoir piloter l'IAG (prompting) | Moy. = 5.313
ρ(Q20 → Q1) = 0.379 | p = 0.002 ** | N = 67

  Q21 Savoir détecter les erreurs de l'IAG | Moy. = 5.254
ρ(Q21 → Q1) = 0.242 | p = 0.048 * | N = 67

→ Résultat clé : Q19 (formation reçue) est NS ; Q20 et Q21 (compétences opérationnelles) sont significatifs


### 5.3.4.2 — H4b : Maîtrise fonctionnelle, risques perçus et scepticisme professionnel

**H4b-1** : ρ(SMF, SSP) > 0 (maîtrise → scepticisme élevé)  
**H4b-2** : ρ(SRP, SSP) < 0 (risques non maîtrisés → érosion du scepticisme)

In [ ]:
# ── H4b : Statistiques descriptives SSP et ses items ────────────────────────
print("=== H4b — Scepticisme professionnel (SSP) ===")
print(f"SSP global : moyenne = {df[SSP].mean():.3f} | médiane = {df[SSP].median():.1f} | σ = {df[SSP].std():.3f}")

ssp_items_desc = [
    (Q22_inv, 'Q22_inv (Ne pas se fier sans vérifier)'),
    (Q23,     'Q23 (Fréquence vérification sources)'),
    (Q24,     'Q24 (Jugement professionnel fort)'),
    (Q25_inv, 'Q25_inv (Influence IAG inversée)'),
    (Q26_inv, 'Q26_inv (Ne pas conclure avant preuve)'),
]
for col, label in ssp_items_desc:
    s = pd.to_numeric(df[col], errors='coerce')
    print(f"  {label:<45}: moy. = {s.mean():.3f} | méd. = {s.median():.1f}")

=== H4b — Scepticisme professionnel (SSP) ===
SSP global : moyenne = 4.958 | médiane = 5.0 | σ = 0.793
  Q22_inv (Ne pas se fier sans vérifier)       : moy. = 5.119 | méd. = 5.0
  Q23 (Fréquence vérification sources)         : moy. = 5.478 | méd. = 6.0
  Q24 (Jugement professionnel fort)            : moy. = 5.597 | méd. = 6.0
  Q25_inv (Influence IAG inversée)             : moy. = 3.433 | méd. = 3.0
  Q26_inv (Ne pas conclure avant preuve)       : moy. = 5.164 | méd. = 6.0


In [ ]:
# ── H4b-1 : SMF → SSP ────────────────────────────────────────────────────────
print("=== H4b-1 — SMF → SSP ===")
rho_h4b1, p_h4b1 = spearman_report(df[SMF], df[SSP], 'SMF', 'SSP')

# ── H4b-2 : SRP → SSP ────────────────────────────────────────────────────────
print("\n=== H4b-2 — SRP → SSP ===")
rho_h4b2, p_h4b2 = spearman_report(df[SRP], df[SSP], 'SRP', 'SSP')

print("\n→ VERDICT : H4b-1 et H4b-2 INFIRMÉES (NS) — faible variance SSP + désirabilité sociale")

=== H4b-1 — SMF → SSP ===
ρ(SMF → SSP) = 0.190 | p = 0.124 NS | N = 67

=== H4b-2 — SRP → SSP ===
ρ(SRP → SSP) = 0.113 | p = 0.364 NS | N = 67

→ VERDICT : H4b-1 et H4b-2 INFIRMÉES (NS) — faible variance SSP + désirabilité sociale


In [ ]:
# ── H4b : Tests désagrégés — signaux du biais d'automatisation ───────────────
print("=== H4b — Signaux du biais d'automatisation (tests désagrégés) ===")

q23_num = pd.to_numeric(df[Q23], errors='coerce')
q24_num = pd.to_numeric(df[Q24], errors='coerce')
q25_inv_num = pd.to_numeric(df[Q25_inv], errors='coerce')

# Effets PROTECTEURS
print("\n-- Effets protecteurs --")
print("  Q21 (détecter erreurs) → SSP (scepticisme)")
spearman_report(q21_num, df[SSP], 'Q21', 'SSP')
print("\n  Q12 (conscience hallucinations) → Q23 (vérification sources)")
spearman_report(df[Q12], q23_num, 'Q12', 'Q23')

# Signaux INQUIÉTANTS (biais d'automatisation)
print("\n-- Signaux inquiétants : biais d'automatisation --")
print("  Q1 (usage intensif) → Q24 (se croit sceptique)")
spearman_report(df[Q1], q24_num, 'Q1', 'Q24')
print("\n  Q1 (usage intensif) → Q25_inv (subit l'IAG — cède au jugement de l'IAG)")
spearman_report(df[Q1], q25_inv_num, 'Q1', 'Q25_inv')

print("\n→ Se croire sceptique (Q24↑) tout en cédant à l'IAG (Q25_inv↓) = définition du biais d'automatisation")

=== H4b — Signaux du biais d'automatisation (tests désagrégés) ===

-- Effets protecteurs --
  Q21 (détecter erreurs) → SSP (scepticisme)
ρ(Q21 → SSP) = 0.278 | p = 0.023 * | N = 67

  Q12 (conscience hallucinations) → Q23 (vérification sources)
ρ(Q12 → Q23) = 0.312 | p = 0.010 * | N = 67

-- Signaux inquiétants : biais d'automatisation --
  Q1 (usage intensif) → Q24 (se croit sceptique)
ρ(Q1 → Q24) = 0.282 | p = 0.021 * | N = 67

  Q1 (usage intensif) → Q25_inv (subit l'IAG — cède au jugement de l'IAG)
ρ(Q1 → Q25_inv) = -0.271 | p = 0.027 * | N = 67

→ Se croire sceptique (Q24↑) tout en cédant à l'IAG (Q25_inv↓) = définition du biais d'automatisation


### 5.3.5 — Tableau récapitulatif des tests d'hypothèses

In [ ]:
# ── Tableau récapitulatif ─────────────────────────────────────────────────────
print("=" * 90)
print(f"{'Hyp.':<6} {'Variables':<25} {'ρ':>7} {'p-value':>9} {'Sig.':<6} {'Verdict':<15} {'Résultat clé'}")
print("=" * 90)

def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    if p < 0.10:  return '(.)'
    return 'NS'

resultats = [
    ('H1',    'SUP → Q1',        rho_h1,  p_h1,  'INFIRMÉE',   'Effet plafond SUP'),
    ('H2',    'SFU → Q1',        rho_h2,  p_h2,  'CONFIRMÉE',  'Q10 workflow ρ≈0.49***'),
    ('H3',    'SRP → Q1',        rho_h3_q1, p_h3_q1, 'CONFIRMÉE', 'Risques freinent l\'usage'),
    ('H3',    'SRP → Q17',       rho_h3_q17, p_h3_q17, 'CONFIRMÉE', 'Q15 Responsabilité = frein'),
    ('H4a',   'Gradient SMF',    np.nan,  np.nan,'TENDANCE',   '-0.416 → -0.274 → 0.030'),
    ('H4b-1', 'SMF → SSP',       rho_h4b1, p_h4b1, 'INFIRMÉE',  'NS — biais automatis. détecté'),
    ('H4b-2', 'SRP → SSP',       rho_h4b2, p_h4b2, 'INFIRMÉE',  'NS — direction opposée'),
]

for hyp, var, rho, p, verdict, note in resultats:
    rho_str = f"{rho:.3f}" if not np.isnan(rho) else '  —  '
    p_str   = f"{p:.3f}"   if not np.isnan(p)   else '  —  '
    sig     = sig_stars(p)  if not np.isnan(p)   else ' — '
    print(f"{hyp:<6} {var:<25} {rho_str:>7} {p_str:>9} {sig:<6} {verdict:<15} {note}")

print("=" * 90)

Hyp.   Variables                       ρ   p-value Sig.   Verdict         Résultat clé
H1     SUP → Q1                    0.168     0.174 NS     INFIRMÉE        Effet plafond SUP
H2     SFU → Q1                    0.325     0.007 **     CONFIRMÉE       Q10 workflow ρ≈0.49***
H3     SRP → Q1                   -0.231     0.060 (.)    CONFIRMÉE       Risques freinent l'usage
H3     SRP → Q17                   0.310     0.011 *      CONFIRMÉE       Q15 Responsabilité = frein
H4a    Gradient SMF                  —         —    —     TENDANCE        -0.416 → -0.274 → 0.030
H4b-1  SMF → SSP                   0.190     0.124 NS     INFIRMÉE        NS — biais automatis. détecté
H4b-2  SRP → SSP                   0.113     0.364 NS     INFIRMÉE        NS — direction opposée


---
## Section 5.4 — Analyses complémentaires

### 5.4.1 — Classement des risques perçus par importance (Q16)

In [ ]:
# ── Q16 — Importance perçue des risques (échelle 1-5) ────────────────────────
print("=== Q16 — Importance perçue des 4 risques (1-5) ===")
risques_q16 = [
    (Q16_CONFID, 'Confidentialité'),
    (Q16_HALLU,  'Hallucinations'),
    (Q16_RESP,   'Responsabilité'),
    (Q16_BOITE,  'Boîte noire'),
]
for col, label in risques_q16:
    s = df[col]
    pct_max = (s == s.max()).sum()
    print(f"  {label:<20}: moy. = {s.mean():.3f} | méd. = {s.median():.1f} | note max (5/5) : {pct_max} ({pct_max/len(df)*100:.1f} %)")

=== Q16 — Importance perçue des 4 risques (1-5) ===
  Confidentialité     : moy. = 4.433 | méd. = 5.0 | note max (5/5) : 46 (68.7 %)
  Hallucinations      : moy. = 4.343 | méd. = 5.0 | note max (5/5) : 34 (50.7 %)
  Responsabilité      : moy. = 3.881 | méd. = 4.0 | note max (5/5) : 22 (32.8 %)
  Boîte noire         : moy. = 3.642 | méd. = 4.0 | note max (5/5) : 13 (19.4 %)


### 5.4.2 — Intentions d'usage futur et effet levier formation (Q27, Q28)

In [ ]:
# ── Q27 — Intention d'augmenter l'usage ──────────────────────────────────────
print("=== Q27 — Intention d'augmenter l'usage dans les 6 prochains mois ===")
q27_counts = df[Q27].value_counts()
# Oui / Non / Conditionnel
oui = (df[Q27] == 'Oui').sum()
non = (df[Q27] == 'Non').sum()
conditionnel = len(df) - oui - non
print(f"  Oui          : {oui} ({oui/len(df)*100:.1f} %)")
print(f"  Non          : {non} ({non/len(df)*100:.1f} %)")
print(f"  Conditionnel : {conditionnel} ({conditionnel/len(df)*100:.1f} %)")

# ── Q28 — Probabilité d'augmenter si formation ───────────────────────────────
q28_num = pd.to_numeric(df[Q28], errors='coerce')
print(f"\n=== Q28 — Probabilité d'augmenter l'usage si formation (1-7) ===")
print(f"  Moyenne   : {q28_num.mean():.3f}")
print(f"  Médiane   : {q28_num.median():.1f}")
print(f"  σ         : {q28_num.std():.3f}")
for val in [5, 6, 7]:
    n = (q28_num == val).sum()
    print(f"  Score {val}    : {n} ({n/len(df)*100:.1f} %)")
bas_q28 = (q28_num <= 3).sum()
print(f"  Scores ≤ 3 : {bas_q28} ({bas_q28/len(df)*100:.1f} %)")

=== Q27 — Intention d'augmenter l'usage dans les 6 prochains mois ===
  Oui          : 39 (58.2 %)
  Non          : 24 (35.8 %)
  Conditionnel : 4 (6.0 %)

=== Q28 — Probabilité d'augmenter l'usage si formation (1-7) ===
  Moyenne   : 5.194
  Médiane   : 5.0
  σ         : 1.708
  Score 5    : 17 (25.4 %)
  Score 6    : 15 (22.4 %)
  Score 7    : 18 (26.9 %)
  Scores ≤ 3 : 10 (14.9 %)


### 5.4.3 — Les risques comme verrou réversible : corrélation Q17 – Q18

In [ ]:
# ── Q17 vs Q18 — Frein actuel vs Intention si risques éliminés ───────────────
q18_num = pd.to_numeric(df[Q18], errors='coerce')

print("=== Q17 vs Q18 — Frein actuel vs Intention latente ===")
print(f"  Q17 (frein actuel)             : moy. = {q17_num.mean():.2f} | méd. = {q17_num.median():.1f}")
print(f"  Q18 (si risques éliminés)      : moy. = {q18_num.mean():.2f} | méd. = {q18_num.median():.1f}")
print(f"  Écart Q18 - Q17                : {q18_num.mean() - q17_num.mean():.2f} points")

# Note max (7)
n_max_q17 = (q17_num == 7).sum()
n_max_q18 = (q18_num == 7).sum()
print(f"  Note maximale (7) sur Q17      : {n_max_q17} ({n_max_q17/len(df)*100:.1f} %)")
print(f"  Note maximale (7) sur Q18      : {n_max_q18} ({n_max_q18/len(df)*100:.1f} %)")

print("\n  Corrélation Q17 → Q18 :")
spearman_report(q17_num, q18_num, 'Q17', 'Q18')

=== Q17 vs Q18 — Frein actuel vs Intention latente ===
  Q17 (frein actuel)             : moy. = 4.27 | méd. = 4.0
  Q18 (si risques éliminés)      : moy. = 5.73 | méd. = 6.0
  Écart Q18 - Q17                : 1.46 points
  Note maximale (7) sur Q17      : 6 (9.0 %)
  Note maximale (7) sur Q18      : 28 (41.8 %)

  Corrélation Q17 → Q18 :
ρ(Q17 → Q18) = 0.310 | p = 0.011 * | N = 67


(np.float64(0.30986095476058023), np.float64(0.01071894759649064))

### 5.4.4 — Comparaisons par grade hiérarchique (Kruskal-Wallis)

In [ ]:
# ── Test de Kruskal-Wallis par grade (Junior / Senior-Manager / Associé) ──────
def kruskal_report(col, label, df_=df, grade_col=GRADE):
    """Effectue un test de Kruskal-Wallis entre les 3 grades et affiche le résultat."""
    groups = []
    means  = []
    grades = ['Junior', 'Senior-Manager', 'Associe']
    for g in grades:
        sub = df_[df_[grade_col] == g][col].dropna()
        groups.append(sub)
        means.append(sub.mean())
    H, p = kruskal(*groups)
    sig = sig_stars(p)
    means_str = ' | '.join([f"{g[:5]}={m:.3f}" for g, m in zip(grades, means)])
    print(f"  {label:<10} : H={H:.2f} p={p:.3f} {sig:<5} | {means_str}")
    return H, p

print("=== Kruskal-Wallis — Comparaisons par grade ===")
print(f"  {'Score':<10}   {'H':>5} {'p-value':>9} {'Sig.':<5} | {'Juniors':>10} | {'Sen-Mgr':>10} | {'Associés':>10}")
print("-" * 75)
for score_col, label in [(SUP,'SUP'), (SFU,'SFU'), (SRP,'SRP'),
                          (SMF,'SMF'), (SSP,'SSP'), (Q1,'Q1')]:
    kruskal_report(score_col, label)

=== Kruskal-Wallis — Comparaisons par grade ===
  Score            H   p-value Sig.  |    Juniors |    Sen-Mgr |   Associés
---------------------------------------------------------------------------


  SUP        : H=2.64 p=0.267 NS    | Junio=3.317 | Senio=3.356 | Assoc=3.584
  SFU        : H=9.56 p=0.008 **    | Junio=4.906 | Senio=5.078 | Assoc=4.222
  SRP        : H=3.63 p=0.163 NS    | Junio=5.352 | Senio=5.544 | Assoc=5.861


  SMF        : H=5.47 p=0.065 (.)   | Junio=4.615 | Senio=5.176 | Assoc=4.704
  SSP        : H=1.85 p=0.396 NS    | Junio=4.806 | Senio=5.082 | Assoc=5.111


  Q1         : H=0.67 p=0.717 NS    | Junio=4.938 | Senio=5.235 | Assoc=4.944


In [ ]:
# ── Scores moyens par grade (tableau détaillé) ────────────────────────────────
print("=== Moyennes détaillées par grade ===")
grade_groups = df.groupby(GRADE)[[SUP, SFU, SRP, SMF, SSP, Q1]].mean().round(3)
print(grade_groups.to_string())
print(f"\nN par grade : {df[GRADE].value_counts().to_dict()}")

=== Moyennes détaillées par grade ===


                  SUP    SFU    SRP    SMF     SS  Q1. À quelle fréquence utilisez-vous l'IA générative dans le cadre de votre travail en audit ?  (num)
Q33_groupe                                                                                                                                              
Associe         3.584  4.222  5.861  4.704  5.111                                                                                                  4.944
Junior          3.317  4.906  5.352  4.615  4.806                                                                                                  4.938
Senior-Manager  3.356  5.078  5.544  5.176  5.082                                                                                                  5.235

N par grade : {'Junior': 32, 'Associe': 18, 'Senior-Manager': 17}


### 5.4.5 — Comparaisons cabinets internationaux vs autres (Mann-Whitney)

In [ ]:
# ── Création des deux groupes de cabinets ─────────────────────────────────────
# Groupe 1 : Cabinets internationaux / Big4
# Groupe 2 : Autres (locaux, régionaux, réseau)
intl_mask  = df[CABINET].str.contains('international', case=False, na=False)
autres_mask = ~intl_mask

df_intl   = df[intl_mask]
df_autres = df[autres_mask]

print(f"Cabinets internationaux : N = {len(df_intl)} | Autres : N = {len(df_autres)}")

# ── Mann-Whitney sur chaque score ─────────────────────────────────────────────
print("\n=== Mann-Whitney — Comparaisons par type de cabinet ===")
print(f"  {'Score':<8} {'Moy. Intl':>10} {'Moy. Autres':>12} {'U':>8} {'p-value':>9} {'Sig.':<5}")
print("-" * 60)

for score_col, label in [(SUP,'SUP'), (SFU,'SFU'), (SRP,'SRP'),
                          (SMF,'SMF'), (SSP,'SSP'), (Q1,'Q1')]:
    x = df_intl[score_col].dropna()
    y = df_autres[score_col].dropna()
    U, p = mannwhitneyu(x, y, alternative='two-sided')
    sig = sig_stars(p)
    print(f"  {label:<8} {x.mean():>10.3f} {y.mean():>12.3f} {U:>8.0f} {p:>9.3f} {sig:<5}")

Cabinets internationaux : N = 37 | Autres : N = 30

=== Mann-Whitney — Comparaisons par type de cabinet ===
  Score     Moy. Intl  Moy. Autres        U   p-value Sig. 
------------------------------------------------------------
  SUP           3.299        3.522      460     0.231 NS   
  SFU           4.937        4.556      694     0.077 (.)  
  SRP           5.338        5.783      412     0.069 (.)  
  SMF           4.721        4.856      510     0.572 NS   
  SSP           4.811        5.140      435     0.130 NS   
  Q1            5.054        4.967      578     0.773 NS   


---
## Récapitulatif général

Ce notebook reproduit l'intégralité des chiffres de la partie empirique (Chapitres 4–5) :

| Section | Contenu |
|---------|--------|
| 5.1.1 | Entonnoir de filtrage (N = 85 → 67) |
| 5.1.2 | Profil sociodémographique (genre, âge, grade, cabinet, ancienneté) |
| 5.1.3 | Usage des outils IAG (Q2, Q3, Q4, Q5) |
| 5.2.1 | Distribution de Q1 (moy. ≈ 5,015, médiane = 5) |
| 5.2.2 | Statistiques des 5 scores composites + Alpha de Cronbach |
| 5.2.3 | Équilibre bénéfices-risques Q30 |
| 5.3.1 | H1 INFIRMÉE — effet plafond SUP | ρ = 0,168 NS |
| 5.3.2 | H2 CONFIRMÉE (**) — Q10 = corrélation la plus forte (ρ ≈ 0,492) |
| 5.3.3 | H3 CONFIRMÉE (.) — Q15 responsabilité = frein principal |
| 5.3.4.1 | H4a TENDANCE — gradient ρ: -0,416 → -0,274 → 0,030 |
| 5.3.4.2 | H4b INFIRMÉE — mais biais d'automatisation détecté |
| 5.4.1 | Q16 — Confidentialité = risque perçu le plus important |
| 5.4.2 | Q27, Q28 — 58,2 % comptent augmenter leur usage |
| 5.4.3 | Q17–Q18 — risques = verrou réversible (Δ ≈ 1,5 pt) |
| 5.4.4 | Kruskal-Wallis — SFU varie significativement par grade (**) |
| 5.4.5 | Mann-Whitney — pas de différence internationale vs autres sur Q1 |